In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project

Mounted at /content/drive
/content/drive/MyDrive/ml_final_project


In [ ]:
from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline, split_and_save

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q mlflow dagshub pytorch_forecasting lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import pandas as pd
import numpy as np
import torch
import mlflow
import dagshub
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline, GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import warnings
warnings.filterwarnings('ignore')

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [ ]:
dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment("TFT_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d910808a-4167-4a64-875a-66e74caf5812&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7b7fe5f6859574e79ea8a5dba8a4069dd3609dceae704791100c430e45afb052




Accessing as aochi23

Initialized MLflow to track repo "aochi23/ml_final_project"

Repository aochi23/ml_final_project initialized!

<Experiment: artifact_location='mlflow-artifacts:/3f24e094d54c468cbb5545522a77e88a', creation_time=1783785418144, effective_trace_archival_retention=None, experiment_id='9', last_update_time=1783785418144, lifecycle_stage='active', name='TFT_Training', tags={}, trace_location=None, workspace='default'>

In [ ]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)
processed_train, processed_test = split_and_save(full_df)

In [ ]:
print(processed_train.columns.tolist())
print()
print(processed_train.dtypes)
print()
print(processed_train.isnull().sum()[processed_train.isnull().sum() > 0])

['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size', 'markdown_total', 'n_active_markdowns', 'n_weeks', 'is_sparse', 'week_of_year', 'month', 'year', 'days_to_nearest_holiday', 'is_week_after_thanksgiving', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_52', 'lag_53', 'rolling_mean_4', 'rolling_mean_8', 'rolling_std_4', 'rolling_std_8', 'yoy_growth', 'dept_avg_sales', 'store_avg_sales', 'type_ordinal']

Store                                  int64
Dept                                   int64
Date                          datetime64[ns]
Weekly_Sales                         float64
IsHoliday                               bool
Temperature                          float64
Fuel_Price                           float64
MarkDown1                            float64
MarkDown2                            float64
MarkDown3                            float64
MarkDown4     

In [ ]:
processed_train = processed_train.copy()
processed_test = processed_test.copy()

processed_train['Date'] = pd.to_datetime(processed_train['Date'])
processed_test['Date'] = pd.to_datetime(processed_test['Date'])

min_date = processed_train['Date'].min()
processed_train['time_idx'] = ((processed_train['Date'] - min_date).dt.days // 7).astype(int)
processed_test['time_idx'] = ((processed_test['Date'] - min_date).dt.days // 7).astype(int)

for col in ['Store', 'Dept', 'Type']:
    processed_train[col] = processed_train[col].astype(str).astype('category')
    processed_test[col] = processed_test[col].astype(str).astype('category')

processed_train['IsHoliday'] = processed_train['IsHoliday'].astype(str).astype('category')
processed_test['IsHoliday'] = processed_test['IsHoliday'].astype(str).astype('category')

processed_train['wmae_weight'] = np.where(processed_train['IsHoliday'] == 'True', 5.0, 1.0)
processed_test['wmae_weight'] = np.where(processed_test['IsHoliday'] == 'True', 5.0, 1.0)

In [ ]:
DROP_COLS = ['lag_52', 'lag_53', 'yoy_growth', 'n_weeks']

processed_train = processed_train.drop(columns=DROP_COLS)
processed_test = processed_test.drop(columns=[c for c in DROP_COLS if c in processed_test.columns])

LAG_MEAN_COLS = ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'rolling_mean_4', 'rolling_mean_8']
LAG_STD_COLS = ['rolling_std_4', 'rolling_std_8']

for col in LAG_MEAN_COLS:
    processed_train[col] = processed_train[col].fillna(processed_train['dept_avg_sales'])

for col in LAG_STD_COLS:
    processed_train[col] = processed_train[col].fillna(0)

print(processed_train.isnull().sum()[processed_train.isnull().sum() > 0])

Series([], dtype: int64)


In [ ]:
with mlflow.start_run(run_name="TFT_Preprocessing"):
    val_horizon = 39
    last_date = processed_train['Date'].max()
    val_start_date = last_date - pd.Timedelta(weeks=val_horizon - 1)

    train_df = processed_train[processed_train['Date'] < val_start_date].copy()
    val_df = processed_train[processed_train['Date'] >= val_start_date].copy()

    mlflow.log_param("val_horizon_weeks", val_horizon)
    mlflow.log_param("val_start_date", str(val_start_date.date()))
    mlflow.log_param("dropped_cols", str(DROP_COLS))
    mlflow.log_param("n_train_rows", len(train_df))
    mlflow.log_param("n_val_rows", len(val_df))

    print(f"Train: {train_df['Date'].min().date()} to {train_df['Date'].max().date()} ({len(train_df)} rows)")
    print(f"Val:   {val_df['Date'].min().date()} to {val_df['Date'].max().date()} ({len(val_df)} rows)")

Train: 2010-02-05 to 2012-01-27 (305982 rows)
Val:   2012-02-03 to 2012-10-26 (115588 rows)
🏃 View run TFT_Preprocessing at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/7663b58a29b7460ca820a67d0d3d2579
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9


In [ ]:
val_horizon = 39
last_date = processed_train['Date'].max()
val_start_date = last_date - pd.Timedelta(weeks=val_horizon - 1)

train_df = processed_train[processed_train['Date'] < val_start_date].copy()
val_df = processed_train[processed_train['Date'] >= val_start_date].copy()

print(f"Train: {train_df['Date'].min().date()} to {train_df['Date'].max().date()} ({len(train_df)} rows)")
print(f"Val:   {val_df['Date'].min().date()} to {val_df['Date'].max().date()} ({len(val_df)} rows)")

Train: 2010-02-05 to 2012-01-27 (305982 rows)
Val:   2012-02-03 to 2012-10-26 (115588 rows)


In [ ]:
max_encoder_length = 52
max_prediction_length = 39

known_reals = [
    "time_idx", "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "markdown_total", "n_active_markdowns",
    "week_of_year", "month", "year",
    "days_to_nearest_holiday", "is_week_after_thanksgiving",
]

unknown_reals = [
    "Weekly_Sales",
    "lag_1", "lag_2", "lag_3", "lag_4",
    "rolling_mean_4", "rolling_mean_8", "rolling_std_4", "rolling_std_8",
]

In [ ]:
training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["Store", "Dept"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    static_categoricals=["Store", "Dept", "Type"],
    static_reals=["Size", "dept_avg_sales", "store_avg_sales", "type_ordinal"],

    time_varying_known_categoricals=["IsHoliday"],
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=unknown_reals,

    weight="wmae_weight",
    target_normalizer=GroupNormalizer(groups=["Store", "Dept"]),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

In [ ]:
training.save(f"{drive_repo}/tft_training_dataset.pkl")
print("Saved TimeSeriesDataSet definition for reuse in model_inference.ipynb")

Saved TimeSeriesDataSet definition for reuse in model_inference.ipynb


In [ ]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    pd.concat([train_df, val_df], axis=0).reset_index(drop=True),
    predict=True,
    stop_randomization=True,
)

print(f"Training samples: {len(training)}")
print(f"Validation samples: {len(validation)}")

Training samples: 40136
Validation samples: 2938


In [ ]:
BATCH_SIZE = 128

train_loader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=2)
val_loader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=2)

In [ ]:
class WMAECallback(pl.Callback):
    def __init__(self):
        super().__init__()
        self.num = []
        self.den = []

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        target = y[0]
        weights = y[1] if (len(y) > 1 and y[1] is not None) else torch.ones_like(target)

        with torch.no_grad():
            out = pl_module(x)
        pred = pl_module.to_prediction(out)

        weights = weights.to(pred.device)
        target = target.to(pred.device)

        self.num.append((weights * (pred - target).abs()).sum().detach())
        self.den.append(weights.sum().detach())

    def on_validation_epoch_end(self, trainer, pl_module):
        score = torch.stack(self.num).sum() / torch.stack(self.den).sum()
        pl_module.log("val_wmae", score, prog_bar=True, logger=True)
        self.num.clear()
        self.den.clear()

In [ ]:
with mlflow.start_run(run_name="TFT_Baseline"):
    baseline_predictions = Baseline().predict(
        val_loader,
        return_x=True,
        return_y=True,
        return_index=True,
    )

    y_pred = baseline_predictions.output.cpu()
    y_true = baseline_predictions.y[0].cpu()
    weights = baseline_predictions.y[1].cpu() if baseline_predictions.y[1] is not None else torch.ones_like(y_true)

    baseline_wmae = ((weights * (y_pred - y_true).abs()).sum() / weights.sum()).item()

    mlflow.log_param("model_type", "naive_baseline")
    mlflow.log_metric("val_wmae", baseline_wmae)

    print(f"Baseline (naive last-value) WMAE: {baseline_wmae:.2f}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Baseline (naive last-value) WMAE: 3217.57
🏃 View run TFT_Baseline at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/ec7d1cb5f2e948caa86f4363b05c9e91
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9


In [ ]:
from pytorch_forecasting.metrics import QuantileLoss

tft_params = {
    "learning_rate": 0.03,
    "hidden_size": 16,
    "attention_head_size": 1,
    "dropout": 0.1,
    "hidden_continuous_size": 8,
    "max_epochs": 30,
}

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=tft_params["learning_rate"],
    hidden_size=tft_params["hidden_size"],
    attention_head_size=tft_params["attention_head_size"],
    dropout=tft_params["dropout"],
    hidden_continuous_size=tft_params["hidden_continuous_size"],
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

print(f"Number of parameters: {tft.size()/1e3:.1f}k")

Number of parameters: 53.5k


In [ ]:
with mlflow.start_run(run_name="TFT_Training_v1"):
    mlflow.log_params(tft_params)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("max_encoder_length", max_encoder_length)
    mlflow.log_param("max_prediction_length", max_prediction_length)

    early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min")
    wmae_callback = WMAECallback()

    trainer = pl.Trainer(
        max_epochs=tft_params["max_epochs"],
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        gradient_clip_val=0.1,
        callbacks=[early_stop_callback, wmae_callback],
        enable_progress_bar=True,
        logger=False,
    )

    trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

    final_val_wmae = trainer.callback_metrics.get("val_wmae")
    if final_val_wmae is not None:
        mlflow.log_metric("val_wmae", final_val_wmae.item())
    mlflow.log_metric("epochs_trained", trainer.current_epoch)

    print(f"Final val_wmae: {final_val_wmae.item() if final_val_wmae is not None else 'N/A'}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  1.9 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    544 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  5.1 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 19.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 13.2 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  1.1 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 53.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 53.5 K                                                                                               
Total estimated model params size (MB): 0.214                                                                      
Modules in train mode: 940                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Final val_wmae: 2859.132568359375
🏃 View run TFT_Training_v1 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/825251f479504af3903c49980e7289f2
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9


In [ ]:
from itertools import product
from lightning.pytorch.callbacks import ModelCheckpoint

sweep_configs = [
    {"hidden_size": 32, "hidden_continuous_size": 16, "learning_rate": 0.01, "dropout": 0.2},
    {"hidden_size": 64, "hidden_continuous_size": 32, "learning_rate": 0.01, "dropout": 0.2},
    {"hidden_size": 64, "hidden_continuous_size": 32, "learning_rate": 0.005, "dropout": 0.3},
]

sweep_results_tft = []

with mlflow.start_run(run_name="TFT_HPO_sweep"):
    for i, cfg in enumerate(sweep_configs):
        with mlflow.start_run(run_name=f"TFT_HPO_trial{i}", nested=True):
            mlflow.log_params(cfg)

            tft_trial = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=cfg["learning_rate"],
                hidden_size=cfg["hidden_size"],
                attention_head_size=1,
                dropout=cfg["dropout"],
                hidden_continuous_size=cfg["hidden_continuous_size"],
                loss=QuantileLoss(),
                reduce_on_plateau_patience=4,
            )

            early_stop = EarlyStopping(monitor="val_wmae", patience=5, mode="min")
            wmae_cb = WMAECallback()
            ckpt_cb = ModelCheckpoint(monitor="val_wmae", mode="min", save_top_k=1)

            trainer = pl.Trainer(
                max_epochs=30,
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                devices=1,
                gradient_clip_val=0.1,
                callbacks=[early_stop, wmae_cb, ckpt_cb],
                enable_progress_bar=False,
                logger=False,
            )

            trainer.fit(tft_trial, train_dataloaders=train_loader, val_dataloaders=val_loader)

            trial_wmae = ckpt_cb.best_model_score.item() if ckpt_cb.best_model_score else float('nan')
            mlflow.log_metric("val_wmae", trial_wmae)
            sweep_results_tft.append({**cfg, "val_wmae": trial_wmae, "checkpoint": ckpt_cb.best_model_path})
            print(f"Trial {i}: val_wmae={trial_wmae:.2f}, params={cfg}")

best_tft_config = min(sweep_results_tft, key=lambda r: r['val_wmae'])
print(f"\nBest TFT config: {best_tft_config}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  2.1 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.1 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 14.9 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 65.0 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 40.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  4.2 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    231 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 176 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 176 K                                                                                                
Total estimated model params size (MB): 0.705                                                                      
Modules in train mode: 941                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Trial 0: val_wmae=2957.15, params={'hidden_size': 32, 'hidden_continuous_size': 16, 'learning_rate': 0.01, 'dropout': 0.2}
🏃 View run TFT_HPO_trial0 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/07487264e1b6492aabfeb4d6147a1be9
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Trial 1: val_wmae=2712.17, params={'hidden_size': 64, 'hidden_continuous_size': 32, 'learning_rate': 0.01, 'dropout': 0.2}
🏃 View run TFT_HPO_trial1 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/10c0f5a6181646dcb71e5c783e6fe668
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  2.1 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  2.2 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 50.6 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  210 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  134 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 16.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 610 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 610 K                                                                                                
Total estimated model params size (MB): 2.440                                                                      
Modules in train mode: 941                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Trial 2: val_wmae=2571.43, params={'hidden_size': 64, 'hidden_continuous_size': 32, 'learning_rate': 0.005, 'dropout': 0.3}
🏃 View run TFT_HPO_trial2 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/9d4730a968d64a0f8bf945a7d22452f8
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9
🏃 View run TFT_HPO_sweep at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/5f044785bde04b78a3cb85aa57cfc353
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9

Best TFT config: {'hidden_size': 64, 'hidden_continuous_size': 32, 'learning_rate': 0.005, 'dropout': 0.3, 'val_wmae': 2571.425537109375, 'checkpoint': '/content/drive/MyDrive/ml_final_project/checkpoints/epoch=3-step=1252.ckpt'}


In [ ]:
with mlflow.start_run(run_name="TFT_Final"):
    mlflow.log_params({k: v for k, v in best_tft_config.items() if k not in ['val_wmae', 'checkpoint']})
    mlflow.log_metric("val_wmae", best_tft_config['val_wmae'])
    mlflow.log_param("selected_from_sweep", "TFT_HPO_sweep")
    mlflow.log_param("checkpoint_source", best_checkpoint_path)

    mlflow.pytorch.log_model(
        tft_final,
        name="tft_model",
        registered_model_name="TFT_Pipeline",
        serialization_format="pickle",
    )

    print(f"Registered TFT model — val_wmae={best_tft_config['val_wmae']:.2f}")

2026/07/11 22:03:58 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/11 22:03:58 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/11 22:04:19 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

Registered TFT model — val_wmae=2571.43
🏃 View run TFT_Final at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9/runs/f39a87cebca04c05bff361c3ffc2ed51
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/9
